# Análise de Perfil Energético — Notebook Principal

Este notebook cobre o pipeline completo do projeto: processamento dos dados reais da PPH, exploração (EDA), definição das categorias de eficiência, treinamento do classificador (Random Forest) e geração de recomendações via LLM (Groq).

## 1. Importação de bibliotecas

In [1]:
#%pip install pandas numpy scikit-learn joblib groq -q
import random
import pandas as pd
import numpy as np
from groq import Groq
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib
import re

## 2. Dados

Os dados vêm da PPH (Pesquisa de Posse de Equipamentos e Hábitos de Uso), um levantamento real de consumo energético domiciliar. A partir do arquivo bruto (`consumo_energetico_completo.csv`), processamos e reduzimos as colunas para um conjunto enxuto e relevante para o modelo: `entrevista`, `regiao`, `uf`, `municipio`, `uso_horario_pico` (gerado), `horas_alto_consumo` (gerado), `consumo_total_kwh_mes`, `valor_estimado_conta_reais`, `produtos_maior_consumo` (top 3 aparelhos por kWh), `categoria_maior_consumo`, `quantidade_equipamentos` (soma de todos os aparelhos) e `tipo_imovel` (gerado: Casa, Apartamento ou Comercio).

A célula abaixo processa o dataset bruto da PPH, reduzindo as dezenas de colunas de posse/consumo por aparelho para as colunas finais que o restante do notebook utiliza. É a única célula que depende do arquivo `consumo_energetico_completo.csv` — o restante do pipeline trabalha sobre `base_energetica.csv`, já processado.

In [2]:
"""
Processa o dataset bruto da PPH (consumo_energetico_completo.csv), reduzindo as
dezenas de colunas de posse/consumo por aparelho para um conjunto enxuto e
relevante ao modelo. Para cada domicílio, calcula:
- os 3 aparelhos que mais consumiram energia (produtos_maior_consumo)
- a categoria de maior consumo (categoria_maior_consumo)
- a quantidade total de equipamentos (soma de todos os qtd_*)
Também gera uso_horario_pico, horas_alto_consumo e tipo_imovel, que não existem
no dataset original.
"""

CAMINHO_BRUTO = "C:\\Users\\Usuario\\Desktop\\NOTEBOOK UPDATES\\G9-BR-TEAM-18\\data\\processed\\pph\\consumo_energetico_completo.csv"
CAMINHO_SAIDA = "ml_processed.csv"

# Mapeamento de cada aparelho para sua categoria de consumo
APARELHO_CATEGORIA = {
    "geladeira": "Refrigeração", "freezer": "Refrigeração", "frigobar": "Refrigeração", "bebedouro": "Refrigeração",
    "ar_condicionado": "Climatização", "ar_condicionado_split": "Climatização", "ventilador": "Climatização",
    "ventilador_teto": "Climatização", "aquecedor_eletrico": "Climatização",
    "tv": "Tecnologia", "computador_desktop": "Tecnologia", "notebook": "Tecnologia",
    "roteador_wifi": "Tecnologia", "videogame": "Tecnologia", "caixa_som": "Tecnologia",
    "lampada_led": "Iluminação", "lampada_fluorescente": "Iluminação", "lampada_incandescente": "Iluminação",
    "chuveiro_eletrico": "Eletrodomésticos", "maquina_lavar": "Eletrodomésticos", "secadora_roupas": "Eletrodomésticos",
    "lava_loucas": "Eletrodomésticos", "microondas": "Eletrodomésticos", "forno_eletrico": "Eletrodomésticos",
    "fogao_eletrico": "Eletrodomésticos", "air_fryer": "Eletrodomésticos", "cafeteira_eletrica": "Eletrodomésticos",
    "ferro_passar": "Eletrodomésticos", "aspirador_po": "Eletrodomésticos", "secador_cabelo": "Eletrodomésticos",
    "liquidificador": "Eletrodomésticos", "batedeira": "Eletrodomésticos", "maquina_costura": "Eletrodomésticos",
    "bomba_dagua": "Serviços", "portao_eletrico": "Serviços", "motor_piscina": "Serviços",
}

# Nomes amigáveis para exibição (usados em produtos_maior_consumo e no prompt do LLM)
NOMES_APARELHOS = {
    "geladeira": "Geladeira", "freezer": "Freezer", "frigobar": "Frigobar", "bebedouro": "Bebedouro",
    "ar_condicionado": "Ar-condicionado", "ar_condicionado_split": "Ar-condicionado Split",
    "ventilador": "Ventilador", "ventilador_teto": "Ventilador de Teto", "aquecedor_eletrico": "Aquecedor Elétrico",
    "tv": "TV", "computador_desktop": "Computador Desktop", "notebook": "Notebook",
    "roteador_wifi": "Roteador Wi-Fi", "videogame": "Videogame", "caixa_som": "Caixa de Som",
    "lampada_led": "Lâmpada LED", "lampada_fluorescente": "Lâmpada Fluorescente",
    "lampada_incandescente": "Lâmpada Incandescente", "chuveiro_eletrico": "Chuveiro Elétrico",
    "maquina_lavar": "Máquina de Lavar", "secadora_roupas": "Secadora de Roupas", "lava_loucas": "Lava-louças",
    "microondas": "Micro-ondas", "forno_eletrico": "Forno Elétrico", "fogao_eletrico": "Fogão Elétrico",
    "air_fryer": "Air Fryer", "cafeteira_eletrica": "Cafeteira Elétrica", "ferro_passar": "Ferro de Passar",
    "aspirador_po": "Aspirador de Pó", "secador_cabelo": "Secador de Cabelo", "liquidificador": "Liquidificador",
    "batedeira": "Batedeira", "maquina_costura": "Máquina de Costura", "bomba_dagua": "Bomba d'Água",
    "portao_eletrico": "Portão Elétrico", "motor_piscina": "Motor de Piscina",
}

APARELHOS = list(APARELHO_CATEGORIA.keys())


def calcular_top_produtos(row):
    """Para uma linha (domicílio), retorna os 3 aparelhos de maior consumo (kWh)
    e a categoria do aparelho que mais consumiu."""
    kwh_por_aparelho = {a: row[f"kwh_{a}"] for a in APARELHOS}
    top3 = sorted(kwh_por_aparelho.items(), key=lambda x: x[1], reverse=True)[:3]

    nomes_top3 = [NOMES_APARELHOS[a] for a, v in top3 if v > 0]
    while len(nomes_top3) < 3:
        nomes_top3.append("N/A")

    categoria_top = APARELHO_CATEGORIA[top3[0][0]] if top3[0][1] > 0 else "Outros"

    return pd.Series({
        "produtos_maior_consumo": "|".join(nomes_top3),
        "categoria_maior_consumo": categoria_top,
    })


def gerar_tipo_imovel():
    """Não existe no dataset da PPH — gerado com pesos razoáveis (mais residências
    que comércios, refletindo a composição típica de uma amostra domiciliar)."""
    return random.choices(["Casa", "Apartamento", "Comercio"], weights=[0.45, 0.35, 0.20], k=1)[0]


def gerar_uso_horario_pico():
    """Não existe no dataset da PPH — gerado aleatoriamente (50/50)."""
    return random.random() < 0.5


def gerar_horas_alto_consumo():
    """Não existe no dataset da PPH — gerado aleatoriamente entre 0 e 12 horas."""
    return round(random.uniform(0, 12), 1)


def processar_base(caminho_bruto=CAMINHO_BRUTO, caminho_saida=CAMINHO_SAIDA):
    df_bruto = pd.read_csv(caminho_bruto)

    top_info = df_bruto.apply(calcular_top_produtos, axis=1)
    df_bruto["quantidade_equipamentos"] = df_bruto[[f"qtd_{a}" for a in APARELHOS]].sum(axis=1)

    df_final = pd.concat([
        df_bruto[["ENTREVISTA", "REGIAO", "UF", "MUNICIPIO", "consumo_total_kwh_mes",
                   "valor_estimado_conta_reais", "quantidade_equipamentos"]],
        top_info,
    ], axis=1)

    df_final.columns = [c.lower() for c in df_final.columns]

    df_final["tipo_imovel"] = [gerar_tipo_imovel() for _ in range(len(df_final))]
    df_final["uso_horario_pico"] = [gerar_uso_horario_pico() for _ in range(len(df_final))]
    df_final["horas_alto_consumo"] = [gerar_horas_alto_consumo() for _ in range(len(df_final))]

    ordem_colunas = [
        "entrevista", "regiao", "uf", "municipio", "uso_horario_pico", "horas_alto_consumo",
        "consumo_total_kwh_mes", "valor_estimado_conta_reais", "produtos_maior_consumo",
        "categoria_maior_consumo", "quantidade_equipamentos", "tipo_imovel",
    ]
    df_final = df_final[ordem_colunas]

    df_final.to_csv(caminho_saida, index=False)
    print(f"Base processada com {len(df_final)} registros em '{caminho_saida}'")
    return df_final


if __name__ == "__main__":
    processar_base()

Base processada com 27826 registros em 'ml_processed.csv'


Com a base processada e salva, carregamos o arquivo para dar início à exploração e modelagem.

In [3]:
df = pd.read_csv('ml_processed.csv')

In [4]:
df["uso_horario_pico"] = df["uso_horario_pico"].astype(bool)
df.head()

,entrevista,regiao,uf,municipio,uso_horario_pico,horas_alto_consumo,consumo_total_kwh_mes,valor_estimado_conta_reais,produtos_maior_consumo,categoria_maior_consumo,quantidade_equipamentos,tipo_imovel
0,52641,Norte,RR,Boa Vista,False,3.7,59.01,44.26,Geladeira|Lâmpada Fluorescente|Ventilador,Refrigeração,30,Apartamento
1,52642,Norte,RR,Boa Vista,True,5.3,82.31,61.73,Geladeira|Notebook|Lâmpada LED,Refrigeração,53,Comercio
2,52643,Norte,RR,Boa Vista,True,11.6,53.86,40.39,Geladeira|Lâmpada Fluorescente|Lâmpada LED,Refrigeração,28,Apartamento
3,52647,Norte,RR,Boa Vista,True,11.5,109.23,81.92,Geladeira|Ventilador|Ventilador de Teto,Refrigeração,16,Apartamento
4,52649,Norte,RR,Boa Vista,True,5.8,62.65,46.99,Geladeira|Ventilador|Lâmpada Fluorescente,Refrigeração,18,Casa


## 3. Exploração e limpeza dos dados

Antes de treinar qualquer modelo, verificamos a integridade da base (dimensões, tipos, nulos, duplicatas) e olhamos a distribuição das variáveis mais relevantes. Isso ajuda a validar que os dados gerados (`uso_horario_pico`, `horas_alto_consumo`, `tipo_imovel`) e os dados reais (`consumo_total_kwh_mes`, `categoria_maior_consumo`) estão coerentes antes de seguir.

In [5]:
print("Dimensões:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)
print("\nValores nulos por coluna:")
print(df.isnull().sum())
print("\nRegistros duplicados:", df.duplicated().sum())

print("\nInformações adicionais:")
q33 = np.percentile(df['consumo_total_kwh_mes'], 33)
q66 = np.percentile(df['consumo_total_kwh_mes'], 66)
consumo_minimo = df['consumo_total_kwh_mes'].min()
consumo_maximo = df['consumo_total_kwh_mes'].max()
print('Consumo Minimo: ', consumo_minimo)
print('Consumo Maximo: ', consumo_maximo)
print('Percentil 33: ', q33)
print('Percentil 66: ', q66)

Dimensões: (27826, 12)

Tipos de dados:
entrevista                      int64
regiao                            str
uf                                str
municipio                         str
uso_horario_pico                 bool
horas_alto_consumo            float64
consumo_total_kwh_mes         float64
valor_estimado_conta_reais    float64
produtos_maior_consumo            str
categoria_maior_consumo           str
quantidade_equipamentos         int64
tipo_imovel                       str
dtype: object

Valores nulos por coluna:
entrevista                    0
regiao                        0
uf                            0
municipio                     0
uso_horario_pico              0
horas_alto_consumo            0
consumo_total_kwh_mes         0
valor_estimado_conta_reais    0
produtos_maior_consumo        0
categoria_maior_consumo       0
quantidade_equipamentos       0
tipo_imovel                   0
dtype: int64

Registros duplicados: 2

Informações adicionais:
Consumo Minimo: 

In [6]:
# Variáveis numéricas
print("Estatísticas descritivas")
display(df[["consumo_total_kwh_mes", "quantidade_equipamentos", "horas_alto_consumo"]].describe())

# Variáveis categóricas/booleanas
print("\nDistribuição de tipo_imovel")
print(df["tipo_imovel"].value_counts())
print("\nDistribuição de categoria_maior_consumo")
print(df["categoria_maior_consumo"].value_counts())
print("\nDistribuição de uso_horario_pico")
print(df["uso_horario_pico"].value_counts(normalize=False))

Estatísticas descritivas


,consumo_total_kwh_mes,quantidade_equipamentos,horas_alto_consumo
count,27826.000000,27826.000000,27826.000000
mean,131.763669,11.414181,5.993998
std,167.816045,9.803946,3.469614
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,3.000000
50%,99.500000,12.000000,6.000000
75%,174.737500,18.000000,9.000000
max,3087.280000,95.000000,12.000000



Distribuição de tipo_imovel
tipo_imovel
Casa           12504
Apartamento     9748
Comercio        5574
Name: count, dtype: int64

Distribuição de categoria_maior_consumo
categoria_maior_consumo
Refrigeração        9487
Outros              9051
Eletrodomésticos    4118
Climatização        3737
Tecnologia          1000
Serviços             268
Iluminação           165
Name: count, dtype: int64

Distribuição de uso_horario_pico
uso_horario_pico
True     14082
False    13744
Name: count, dtype: int64


## 4. Definição de categorias

Como o dataset da PPH não vem com um rótulo de eficiência energética, criamos um **índice de ineficiência composto**, combinando consumo (normalizado pelo tipo de imóvel), uso em horário de pico, quantidade de equipamentos e horas de alto consumo. O índice é então cortado em **quintis**, gerando 5 categorias balanceadas: `Excelente`, `Bom`, `Mediano`, `Ruim` e `Crítico`. Esse rótulo é o que o classificador aprenderá a prever a partir dos dados brutos, sem precisar recalcular o índice.

In [8]:
"""
1. Rotula a base real (PPH) criando um índice de ineficiência energética
   e cortando-o em quintis (Excelente / Bom / Mediano / Ruim / Crítico).
2. Salva a base rotulada para uso no treinamento do classificador.
"""

CAMINHO_ENTRADA = "C:\\Users\\Usuario\\Desktop\\NOTEBOOK UPDATES\\G9-BR-TEAM-18\\data\\processed\\for_ml\\ml_processed.csv"
CAMINHO_SAIDA = "rotuled_ml_processed.csv"

COLUNAS_NUMERICAS = ["consumo_total_kwh_mes", "quantidade_equipamentos", "horas_alto_consumo"]
COLUNAS_CATEGORICAS = ["tipo_imovel"]
COLUNAS_BOOLEANAS = ["uso_horario_pico"]

# Consumo médio de referência por tipo de imóvel (usado para normalizar o consumo
# relativo -- evita que Comercio pareça sempre "ineficiente" só por ter escala maior)
CONSUMO_BASE_POR_TIPO = {
    "Casa": 200,
    "Apartamento": 130,
    "Comercio": 600,
}


def calcular_indice_ineficiencia(df):
    """Índice composto (0 a 1) combinando as variáveis de entrada disponíveis."""
    consumo_relativo = df["consumo_total_kwh_mes"] / df["tipo_imovel"].map(CONSUMO_BASE_POR_TIPO)
    consumo_norm = (consumo_relativo / consumo_relativo.max()).clip(0, 1)
    equip_norm = (df["quantidade_equipamentos"] / df["quantidade_equipamentos"].max()).clip(0, 1)
    horas_norm = (df["horas_alto_consumo"] / df["horas_alto_consumo"].max()).clip(0, 1)
    pico_norm = df["uso_horario_pico"].astype(int)

    indice = (
        0.40 * consumo_norm
        + 0.25 * pico_norm
        + 0.20 * equip_norm
        + 0.15 * horas_norm
    )
    return indice


def rotular_por_quintis(indice):
    """Corta o índice em 5 faixas com quantidades iguais de cada classe."""
    return pd.qcut(
        indice,
        q=5,
        labels=["Excelente", "Bom", "Mediano", "Ruim", "Crítico"]
    )


def main():
    df = pd.read_csv(CAMINHO_ENTRADA)
    df["uso_horario_pico"] = df["uso_horario_pico"].astype(int)

    indice = calcular_indice_ineficiencia(df)
    df["categoria"] = rotular_por_quintis(indice)
    df.to_csv(CAMINHO_SAIDA, index=False)

    print("Distribuição das categorias:")
    print(df["categoria"].value_counts(), "\n")


if __name__ == "__main__":
    main()

Distribuição das categorias:
categoria
Mediano      5581
Excelente    5566
Bom          5565
Crítico      5565
Ruim         5549
Name: count, dtype: int64 



Recarregamos a base já rotulada e fazemos um pequeno ajuste de formatação nos valores numéricos antes de seguir para o treinamento.

In [9]:
df = pd.read_csv("C:\\Users\\Usuario\\Desktop\\NOTEBOOK UPDATES\\G9-BR-TEAM-18\\data\\processed\\for_ml\\rotuled_ml_processed.csv")
df.head()

,entrevista,regiao,uf,municipio,uso_horario_pico,horas_alto_consumo,consumo_total_kwh_mes,valor_estimado_conta_reais,produtos_maior_consumo,categoria_maior_consumo,quantidade_equipamentos,tipo_imovel,categoria
0,52641,Norte,RR,Boa Vista,0,3.7,59.01,44.26,Geladeira|Lâmpada Fluorescente|Ventilador,Refrigeração,30,Apartamento,Bom
1,52642,Norte,RR,Boa Vista,1,5.3,82.31,61.73,Geladeira|Notebook|Lâmpada LED,Refrigeração,53,Comercio,Crítico
2,52643,Norte,RR,Boa Vista,1,11.6,53.86,40.39,Geladeira|Lâmpada Fluorescente|Lâmpada LED,Refrigeração,28,Apartamento,Crítico
3,52647,Norte,RR,Boa Vista,1,11.5,109.23,81.92,Geladeira|Ventilador|Ventilador de Teto,Refrigeração,16,Apartamento,Crítico
4,52649,Norte,RR,Boa Vista,1,5.8,62.65,46.99,Geladeira|Ventilador|Lâmpada Fluorescente,Refrigeração,18,Casa,Ruim


In [10]:
# valor_estimado_conta_reais já vem calculado no dataset real da PPH -- não é
# necessário recalcular. Apenas arredondamos consumo_total_kwh_mes para leitura.
df['consumo_total_kwh_mes'] = df['consumo_total_kwh_mes'].round(1)
df['valor_estimado_conta_reais'] = df['valor_estimado_conta_reais'].round(2)
df.head()

,entrevista,regiao,uf,municipio,uso_horario_pico,horas_alto_consumo,consumo_total_kwh_mes,valor_estimado_conta_reais,produtos_maior_consumo,categoria_maior_consumo,quantidade_equipamentos,tipo_imovel,categoria
0,52641,Norte,RR,Boa Vista,0,3.7,59.0,44.26,Geladeira|Lâmpada Fluorescente|Ventilador,Refrigeração,30,Apartamento,Bom
1,52642,Norte,RR,Boa Vista,1,5.3,82.3,61.73,Geladeira|Notebook|Lâmpada LED,Refrigeração,53,Comercio,Crítico
2,52643,Norte,RR,Boa Vista,1,11.6,53.9,40.39,Geladeira|Lâmpada Fluorescente|Lâmpada LED,Refrigeração,28,Apartamento,Crítico
3,52647,Norte,RR,Boa Vista,1,11.5,109.2,81.92,Geladeira|Ventilador|Ventilador de Teto,Refrigeração,16,Apartamento,Crítico
4,52649,Norte,RR,Boa Vista,1,5.8,62.6,46.99,Geladeira|Ventilador|Lâmpada Fluorescente,Refrigeração,18,Casa,Ruim


## 5. Treinamento do modelo de classificação

Comparações anteriores entre Regressão Logística, Random Forest e Árvore de Decisão indicaram o **Random Forest** como melhor opção para esta base (maior acurácia e boa robustez a relações não-lineares entre as variáveis). O pipeline abaixo já inclui o pré-processamento (padronização + one-hot encoding) e salva o modelo treinado para uso posterior, sem necessidade de retreinar a cada análise.

In [11]:
COLUNAS_NUMERICAS = ["consumo_total_kwh_mes", "quantidade_equipamentos", "horas_alto_consumo"]
COLUNAS_CATEGORICAS = ["tipo_imovel"]  # adicione "categoria_maior_consumo" aqui, se for usar como feature
COLUNAS_BOOLEANAS = ["uso_horario_pico"]

pre_processador = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), COLUNAS_NUMERICAS + COLUNAS_BOOLEANAS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), COLUNAS_CATEGORICAS),
    ]
)

pipeline = Pipeline([
    ("pre", pre_processador),
    ("modelo", RandomForestClassifier(n_estimators=200, random_state=42)),
])

# Exclui entrevista/regiao/uf/municipio (identificadores/geografia, sem valor preditivo
# direto para o índice) e valor_estimado_conta_reais (vazamento de dados, já que é
# diretamente proporcional a consumo_total_kwh_mes)
X = df[COLUNAS_NUMERICAS + COLUNAS_CATEGORICAS + COLUNAS_BOOLEANAS]
y = df["categoria"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline.fit(X_treino, y_treino)
y_pred = pipeline.predict(X_teste)

print(f"Acurácia: {accuracy_score(y_teste, y_pred):.3f}\n")
print(classification_report(y_teste, y_pred, zero_division=0))

joblib.dump(pipeline, "modelo_categorizacao.joblib")
print("\nModelo salvo em 'modelo_categorizacao.joblib'")

Acurácia: 0.981

              precision    recall  f1-score   support

         Bom       0.97      0.98      0.97      1113
     Crítico       0.99      0.99      0.99      1113
   Excelente       0.99      0.99      0.99      1113
     Mediano       0.97      0.97      0.97      1117
        Ruim       0.98      0.98      0.98      1110

    accuracy                           0.98      5566
   macro avg       0.98      0.98      0.98      5566
weighted avg       0.98      0.98      0.98      5566


Modelo salvo em 'modelo_categorizacao.joblib'


## 6. Configuração do modelo de linguagem (Groq)

As recomendações textuais são geradas por um LLM hospedado na **Groq**, escolhida pela alta velocidade de inferência (infraestrutura própria em LPU) e por oferecer modelos maiores e mais capazes do que seria viável rodar localmente em CPU. Cole sua chave de API abaixo antes de rodar — nunca commite este notebook com a chave preenchida.

In [12]:
GROQ_API_KEY = "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"
# Insira uma chave de API válida acima.
client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "llama-3.3-70b-versatile"

print("Cliente Groq configurado com o modelo:", MODEL_NAME)

print(f"Modelo carregado: {MODEL_NAME}")

Cliente Groq configurado com o modelo: llama-3.3-70b-versatile
Modelo carregado: llama-3.3-70b-versatile


A função abaixo monta um prompt estruturado (regras + poucos exemplos de estilo) para reduzir alucinações e manter o foco exclusivamente em eficiência elétrica. Ela recebe a categoria já prevista pelo classificador e os aparelhos de maior consumo do domicílio, garantindo que pelo menos uma recomendação seja direcionada a esses aparelhos específicos.

In [13]:
def gerar_recomendacoes(df: dict, categoria: str, max_new_tokens: int = 100) -> list[str]:
    """
    Gera 3 recomendações de eficiência energética com base nos dados de entrada
    e na categoria já calculada pelo classificador (Random Forest).
    """
    system_prompt = (
        "Você é um assistente especializado em eficiência energética residencial e comercial. "
        "Responda sempre em português do Brasil, de forma objetiva e sem rodeios."
    )

    regra_tom = (
        "Tom conforme a categoria: Excelente ou Bom = reforçar boas práticas já adotadas; "
        "Mediano = sugerir ajustes pontuais; Ruim ou Crítico = ser direto sobre a necessidade "
        "de mudança, com mais urgência em Crítico."
    )

    regra_categoria_consumo = (
        f"A categoria de maior consumo do imóvel é '{df['categoria_maior_consumo']}', e os "
        f"aparelhos que mais consomem são: {df['produtos_maior_consumo'].replace('|', ', ')}. "
        f"Pelo menos uma das 3 recomendações deve ser voltada especificamente para reduzir "
        f"ou otimizar o consumo relacionado a esses aparelhos/categoria."
    )

    user_prompt = f"""Com base nos dados abaixo, gere exatamente 3 recomendações curtas, práticas
e realmente úteis para melhorar a eficiência energética do imóvel.

REGRAS OBRIGATÓRIAS:
- Envolva EXCLUSIVAMENTE: hábitos de uso de equipamentos elétricos, horários de consumo, ou
  manutenção/substituição de aparelhos elétricos. Nada de água, gás ou outros recursos.
- {regra_categoria_consumo}
- Baseie-se APENAS nos dados fornecidos. Não invente equipamentos ou hábitos não informados.
- Se o tipo de imóvel for "Apartamento", não sugira painéis solares ou soluções que dependam
  de telhado/espaço externo próprio.
- Cada recomendação deve abordar um aspecto diferente, sem repetir o mesmo tipo de dica.
- Não cite marcas, modelos ou preços. Não use termos técnicos sem explicação simples.
- Máximo 20 palavras por recomendação. Sem emojis, markdown ou numeração.
- {regra_tom}

Exemplo de estilo (não copie o conteúdo):
Reduzir o uso simultâneo de equipamentos durante o horário de pico.
Desligar aparelhos em modo stand-by quando não estiverem em uso.
Distribuir o uso de equipamentos de maior consumo ao longo do dia.

Dados do imóvel:
- Consumo mensal: {df['consumo_total_kwh_mes']} kWh
- Uso em horário de pico: {"Sim" if df['uso_horario_pico'] else "Não"}
- Quantidade de equipamentos: {df['quantidade_equipamentos']}
- Tipo de imóvel: {df['tipo_imovel']}
- Horas de alto consumo por dia: {df['horas_alto_consumo']}
- Categoria de eficiência: {categoria}
- Categoria de maior consumo: {df['categoria_maior_consumo']}
- Aparelhos de maior consumo: {df['produtos_maior_consumo'].replace('|', ', ')}

Responda APENAS com as 3 recomendações, uma por linha, sem numeração,
sem introdução e sem comentários adicionais."""

    mensagens = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    resposta = client.chat.completions.create(
        model=MODEL_NAME,
        messages=mensagens,
        max_tokens=max_new_tokens,
        temperature=0,
    )

    texto = resposta.choices[0].message.content

    recomendacoes = [
        re.sub(r"^\s*[\d]+[\.\)]?\s*", "", linha).strip("-•* ").strip()
        for linha in texto.strip().split("\n")
        if linha.strip()
    ]
    return recomendacoes[:3]

## 7. Teste end-to-end

Simula o fluxo completo que a API vai executar: sorteia um domicílio da base, usa o classificador treinado para prever a categoria de eficiência (sem depender do índice de quintis) e gera as recomendações via Groq.

In [15]:
modelo = joblib.load("C:\\Users\\Usuario\\Desktop\\NOTEBOOK UPDATES\\G9-BR-TEAM-18\\modelo_categorizacao.joblib")

entrevista_id = df["entrevista"].sample(1, random_state=None).values[0]
linha_cliente = df[df["entrevista"] == entrevista_id].iloc[0]
dados_cliente = linha_cliente[["consumo_total_kwh_mes", "uso_horario_pico", "quantidade_equipamentos",
                                 "tipo_imovel", "horas_alto_consumo", "categoria_maior_consumo",
                                 "produtos_maior_consumo"]].to_dict()

# Seleciona só as colunas que o modelo foi treinado para receber
colunas_modelo = ["consumo_total_kwh_mes", "quantidade_equipamentos", "horas_alto_consumo",
                   "tipo_imovel", "uso_horario_pico"]
dados_cliente_df = pd.DataFrame([{k: dados_cliente[k] for k in colunas_modelo}])

# Agora a categoria vem do MODELO, não do índice de quintis
categoria = modelo.predict(dados_cliente_df)[0]
probabilidade = round(modelo.predict_proba(dados_cliente_df).max(), 2)

recomendacoes = gerar_recomendacoes(dados_cliente, categoria)

In [16]:
print(linha_cliente)
print(recomendacoes)
print(f"Categoria prevista: {categoria} (confiança: {probabilidade})")

entrevista                                      60272
regiao                                       Nordeste
uf                                                 AL
municipio                                      Maceió
uso_horario_pico                                    1
horas_alto_consumo                               10.1
consumo_total_kwh_mes                           179.3
valor_estimado_conta_reais                     134.48
produtos_maior_consumo        Ventilador|Geladeira|TV
categoria_maior_consumo                  Climatização
quantidade_equipamentos                            21
tipo_imovel                               Apartamento
categoria                                     Crítico
Name: 5580, dtype: object
['Desligar o ventilador quando não estiver em uso.', 'Reduzir o tempo de uso da TV durante o horário de pico.', 'Verificar a manutenção regular da geladeira para evitar consumo excessivo.']
Categoria prevista: Crítico (confiança: 1.0)


**Nota:** os exemplos de saída abaixo eram de uma versão anterior do pipeline (dados sintéticos, 3 categorias, colunas `id_cliente`/`consumo_kwh`) e não refletem mais a estrutura atual (dados reais da PPH, 5 categorias, `entrevista`/`consumo_total_kwh_mes`). Rode a célula de teste acima para gerar exemplos atualizados.